### Part 10 : End-to-End Multi-Agent Orchestration

##### 1. Notebook Purpose

This notebook connects all previously developed agents into one complete multi-agent workflow.

The workflow:

- Receives the user request.
- Creates the initial shared state.
- Sends the request to the Coordinator Agent.
- Reads the execution plan created by the Coordinator.
- Identifies which tasks are ready.
- Executes the appropriate agents.
- Stores agent results in shared state.
- Continues until all planned tasks are complete.
- Returns the final grounded response.
- 
The orchestrator does not perform SQL analysis, prediction, vector search, retention logic, or response generation itself.

It only coordinates the agents.

##### 2. Technologies Used

- Python
- Databricks
- Pydantic
- TypedDict
- Callable type hints
- Dependency-based task orchestration
- Shared multi-agent state
- Agent runner registry


##### 3. Input

The notebook accepts a natural-language user request.

Example:

``` text

user_request = (
    "For customer 1001, predict churn risk, review similar customer notes, recommend a retention action, and provide a final response."
)
```

##### 4. Output

The workflow returns the completed MultiAgentState.

The final state contains information such as:

``` text

{
    "user_request": ...,
    "execution_plan": ...,
    "agent_results": ...,
    "execution_history": ...,
    "errors": ...,
    "final_response": ...,
}

```

##### 5. Architecture

``` text

User Request
      │
      ▼
create_initial_state()
      │
      ▼
Coordinator Agent
      │
      ▼
Execution Plan
      │
      ▼
Orchestrator
      │
      ├── SQL Agent
      ├── Prediction Agent
      ├── Vector Search Agent
      ├── Retention Agent
      └── Final Response Agent
      │
      ▼
Completed Shared State
      │
      ▼
Final Grounded Response

```

##### 6. Load Shared Components

In [0]:
%run ./01_shared_models_code_only

In [0]:
%run ./02_shared_state_and_helpers_code_only

##### 7. Load Agent Notebooks

In [0]:
%run ./03_coordinator_agent

In [0]:
%run ./04_sql_agent

In [0]:
%run ./05_prediction_agent

In [0]:
%run ./06_vector_search_agent

In [0]:
%run ./07_retention_agent

In [0]:
%run ./08_final_response_agent

##### 8. Imports

In [0]:
from typing import Any, Callable, Dict, List, Tuple
import inspect

##### 9. Verify Agent Function Signatures

In [0]:
#Before creating the registry, verify that the loaded functions match the expected signatures.

print(
    "run_sql_agent:",
    inspect.signature(run_sql_agent),
)

print(
    "run_prediction_agent:",
    inspect.signature(run_prediction_agent),
)

print(
    "run_vector_search_agent:",
    inspect.signature(run_vector_search_agent),
)

print(
    "run_retention_agent:",
    inspect.signature(run_retention_agent),
)

print(
    "run_final_response_agent:",
    inspect.signature(run_final_response_agent),
)

##### 10. Agent Function Types

In [0]:
AgentFunction = Callable[
    ...,
    MultiAgentState,
]

In [0]:
AgentRunnerConfig = Tuple[
    AgentFunction,
    Dict[str, Any],
]

##### 11. Agent Runner Registry

In [0]:
def create_agent_runners(
    sql_analytics_tool: SQLAnalyticsFunction,
    prediction_tool: PredictionToolFunction,
    vector_search_tool: VectorSearchToolFunction,
    retention_tool: RetentionToolFunction,
) -> Dict[str, AgentRunnerConfig]:
    """
    Create the agent-runner registry using the injected tools.
    """

    return {
        SQL_AGENT_NAME: (
            run_sql_agent,
            {
                "sql_analytics_tool": sql_analytics_tool,
            },
        ),

        PREDICTION_AGENT_NAME: (
            run_prediction_agent,
            {
                "prediction_tool": prediction_tool,
            },
        ),

        VECTOR_SEARCH_AGENT_NAME: (
            run_vector_search_agent,
            {
                "vector_search_tool": vector_search_tool,
                "num_results": DEFAULT_NUM_RESULTS,
            },
        ),

        RETENTION_AGENT_NAME: (
            run_retention_agent,
            {
                "retention_tool": retention_tool,
            },
        ),

        FINAL_RESPONSE_AGENT_NAME: (
            run_final_response_agent,
            {},
        ),
    }

##### 12. Check Whether a Task Is Completed

In [0]:
#A task is considered completed when its agent has already stored a result in:

In [0]:
def is_task_completed(
    state: MultiAgentState,
    task: AgentTask,
) -> bool:
    """
    Check whether an agent task has already completed.

    A task is considered completed when its agent name
    exists in state["agent_results"].
    """

    agent_results = state.get(
        "agent_results",
        {},
    )

    return task.agent_name in agent_results

##### 13. Check Whether a Dependency Succeeded

In [0]:
def did_dependency_succeed(
    state: MultiAgentState,
    dependency_name: str,
) -> bool:
    """
    Check whether a dependency has produced a successful result.
    """

    agent_results = state.get(
        "agent_results",
        {},
    )

    dependency_result = agent_results.get(
        dependency_name
    )

    if dependency_result is None:
        return False

    return bool(
        dependency_result.success
    )

##### 14. Check Whether a Task Is Ready

In [0]:
#A task is ready when all agents listed in its depends_on field have completed successfully.
def is_task_ready(
    state: MultiAgentState,
    task: AgentTask,
) -> bool:
    """
    Check whether all dependencies required by a task
    have completed successfully.
    """

    for dependency_name in task.depends_on:

        if not did_dependency_succeed(
            state=state,
            dependency_name=dependency_name,
        ):
            return False

    return True

##### 15. Get All Ready Tasks

In [0]:
def get_ready_tasks(
    state: MultiAgentState,
) -> List[AgentTask]:
    """
    Return all execution-plan tasks that:

    1. Have not already completed.
    2. Have all dependencies satisfied.
    """

    ready_tasks: List[AgentTask] = []

    execution_plan = state.get(
        "execution_plan",
        [],
    )

    for task in execution_plan:

        if is_task_completed(
            state=state,
            task=task,
        ):
            continue

        if is_task_ready(
            state=state,
            task=task,
        ):
            ready_tasks.append(task)

    return ready_tasks

##### 16. Create an Orchestration Error Record

In [0]:
def record_orchestration_error(
    state: MultiAgentState,
    agent_name: str,
    error_type: str,
    error_message: str,
) -> None:
    """
    Create and store an orchestration error.
    """

    error_record = ErrorRecord(
        agent_name=agent_name,
        error_type=error_type,
        error_message=error_message,
    )

    add_error(
        state=state,
        error_record=error_record,
    )

##### 17. Run One Agent

In [0]:
def run_agent(
    state: MultiAgentState,
    task: AgentTask,
    agent_runners: Dict[str, AgentRunnerConfig],
) -> MultiAgentState:
    """
    Execute one agent task using the function and
    keyword arguments stored in AGENT_RUNNERS.
    """

    agent_name = task.agent_name

    if agent_name not in AGENT_RUNNERS:
        error_message = (
            "No agent runner is registered for "
            f"agent '{agent_name}'."
        )

        record_orchestration_error(
            state=state,
            agent_name=agent_name,
            error_type="AgentRegistrationError",
            error_message=error_message,
        )

        raise ValueError(
            error_message
        )

    agent_function, agent_kwargs = AGENT_RUNNERS[
        agent_name
    ]

    try:
        updated_state = agent_function(
            state=state,
            **agent_kwargs,
        )

        return updated_state

    except Exception as exc:
        error_message = (
            f"Agent '{agent_name}' failed during "
            f"orchestration: {str(exc)}"
        )

        record_orchestration_error(
            state=state,
            agent_name=agent_name,
            error_type=type(exc).__name__,
            error_message=error_message,
        )

        raise

##### 18. Check Whether the Workflow Is Complete

In [0]:
def is_workflow_complete(
    state: MultiAgentState,
) -> bool:
    """
    Check whether every task in the execution plan
    has completed.
    """

    execution_plan = state.get(
        "execution_plan",
        [],
    )

    if not execution_plan:
        return False

    return all(
        is_task_completed(
            state=state,
            task=task,
        )
        for task in execution_plan
    )

##### 19. Get Incomplete Task Names

In [0]:
def get_incomplete_agent_names(
    state: MultiAgentState,
) -> List[str]:
    """
    Return the names of agents whose tasks
    have not yet completed.
    """

    execution_plan = state.get(
        "execution_plan",
        [],
    )

    return [
        task.agent_name
        for task in execution_plan
        if not is_task_completed(
            state=state,
            task=task,
        )
    ]

##### 20. Main Multi-Agent Workflow

In [0]:
def run_multi_agent_workflow(
    user_request: str,
    llm_invoke: LLMInvokeFunction,
    agent_runners: Dict[
        str,
        AgentRunnerConfig,
    ],
    max_iterations: int = 20,
) -> MultiAgentState:
    """
    Run the complete dependency-based multi-agent workflow.

    Workflow:
    1. Create the initial shared state.
    2. Run the Coordinator Agent.
    3. Retrieve the generated execution plan.
    4. Find all currently ready tasks.
    5. Execute ready agents sequentially.
    6. Repeat until all tasks are completed.
    """

    if not user_request.strip():
        raise ValueError(
            "The user request cannot be empty."
        )

    if max_iterations <= 0:
        raise ValueError(
            "max_iterations must be greater than zero."
        )

    state = create_initial_state(
        user_request=user_request,
    )

    state = run_coordinator_agent(
        state=state,
        llm_invoke=llm_invoke,
    )

    coordinator_result = state.get(
        "coordinator_result"
    )

    if coordinator_result is None:
        error_message = (
            "The Coordinator Agent did not create "
            "a coordinator result."
        )

        record_agent_error(
            state=state,
            agent_name=COORDINATOR_AGENT_NAME,
            error_code=(
                "MISSING_COORDINATOR_RESULT"
            ),
            error_message=error_message,
        )

        raise ValueError(
            error_message
        )

    execution_plan = (
        coordinator_result.execution_plan
    )

    if not execution_plan:
        error_message = (
            "The Coordinator Agent did not create "
            "an execution plan."
        )

        record_agent_error(
            state=state,
            agent_name=COORDINATOR_AGENT_NAME,
            error_code=(
                "MISSING_EXECUTION_PLAN"
            ),
            error_message=error_message,
        )

        raise ValueError(
            error_message
        )

    iteration = 0

    while not is_workflow_complete(
        state=state,
    ):
        iteration += 1

        if iteration > max_iterations:
            error_message = (
                "The workflow exceeded the maximum "
                f"number of iterations: "
                f"{max_iterations}."
            )

            record_agent_error(
                state=state,
                agent_name=COORDINATOR_AGENT_NAME,
                error_code=(
                    "MAXIMUM_ITERATIONS_EXCEEDED"
                ),
                error_message=error_message,
            )

            raise RuntimeError(
                error_message
            )

        ready_tasks = get_ready_tasks(
            state=state,
        )

        if not ready_tasks:
            incomplete_agents = (
                get_incomplete_agent_names(
                    state=state,
                )
            )

            error_message = (
                "The workflow cannot continue because "
                "no incomplete task is currently ready. "
                "This may indicate a missing dependency, "
                "a failed dependency, or a circular "
                "dependency. "
                f"Incomplete agents: "
                f"{incomplete_agents}"
            )

            record_agent_error(
                state=state,
                agent_name=COORDINATOR_AGENT_NAME,
                error_code=(
                    "BLOCKED_WORKFLOW"
                ),
                error_message=error_message,
            )

            raise RuntimeError(
                error_message
            )

        for task in ready_tasks:
            state = run_agent(
                state=state,
                task=task,
                agent_runners=agent_runners,
            )

    return state

##### 21. How the Main Loop Works

###### Initial state:

state["agent_results"] = {}

The Coordinator creates a plan such as:

``` text

Prediction Agent
    depends_on = []

Vector Search Agent
    depends_on = []

Retention Agent
    depends_on =
        Prediction Agent
        Vector Search Agent

Final Response Agent
    depends_on =
        Retention Agent

```

###### Iteration 1

``` text

ready_tasks = [
    prediction_task,
    vector_search_task,
]

```

The loop executes them sequentially:

``` text

for task in ready_tasks:
    state = run_agent(
        state=state,
        task=task,
    )

```

After execution:

``` text

state["agent_results"] = {
    PREDICTION_AGENT_NAME: prediction_result,
    VECTOR_SEARCH_AGENT_NAME: vector_search_result,
}

```

###### Iteration 2


Retention dependencies are now satisfied:

``` text

ready_tasks = [
    retention_task,
]

```

After Retention executes:

``` text

state["agent_results"] = {
    PREDICTION_AGENT_NAME: prediction_result,
    VECTOR_SEARCH_AGENT_NAME: vector_search_result,
    RETENTION_AGENT_NAME: retention_result,
}
```

###### Iteration 3

``` text

The Final Response task becomes ready:

ready_tasks = [
    final_response_task,
]
```
After it executes, every task is complete and the loop ends.

##### 22. End-to-End Test Helper

In [0]:
def display_workflow_results(
    final_state: MultiAgentState,
) -> None:
    """
    Display the main outputs from a completed workflow.
    """

    print("=" * 100)
    print("USER REQUEST")
    print("=" * 100)
    print(
        final_state.get(
            "user_request",
            "Not available",
        )
    )

    print("\n" + "=" * 100)
    print("EXECUTION PLAN")
    print("=" * 100)

    for task in final_state.get(
        "execution_plan",
        [],
    ):
        print(
            f"Agent: {task.agent_name}"
        )

        print(
            f"Dependencies: {task.depends_on}"
        )

        print("-" * 100)

    print("\n" + "=" * 100)
    print("EXECUTION HISTORY")
    print("=" * 100)

    execution_history = final_state.get(
        "execution_history",
        [],
    )

    if execution_history:
        for record in execution_history:
            print(record)
    else:
        print(
            "No execution-history records found."
        )

    print("\n" + "=" * 100)
    print("AGENT RESULTS")
    print("=" * 100)

    agent_results = final_state.get(
        "agent_results",
        {},
    )

    if agent_results:
        for agent_name, result in agent_results.items():
            print(
                f"\nAgent: {agent_name}"
            )

            print(
                f"Result: {result}"
            )
    else:
        print(
            "No agent results found."
        )

    print("\n" + "=" * 100)
    print("ERRORS")
    print("=" * 100)

    errors = final_state.get(
        "errors",
        [],
    )

    if errors:
        for error in errors:
            print(error)
    else:
        print(
            "No workflow errors."
        )

    print("\n" + "=" * 100)
    print("FINAL RESPONSE")
    print("=" * 100)

    print(
        final_state.get(
            "final_response",
            "No final response was generated.",
        )
    )

#####  23. Test 1 — SQL Analytics Request

In [0]:
def mock_successful_sql_tool(
    question: str,
) -> Dict[str, Any]:
    """
    Return a deterministic SQL result for orchestration testing.
    """

    assert isinstance(question, str)
    assert question.strip()

    return {
        "tool": "sql_analytics_tool",
        "status": "success",
        "sql_action": "count_churned_customers",
        "sql_result": [
            {
                "churned_customers": 1869,
            }
        ],
    }

In [0]:
def mock_sql_coordinator_llm(
    prompt: str,
) -> str:
    assert isinstance(prompt, str)
    assert prompt.strip()

    response = {
        "agent_name": "coordinator_agent",
        "status": "success",
        "message": (
            "Coordinator execution plan created."
        ),
        "task_description": None,
        "error": None,
        "request_type": "sql_analytics",
        "reasoning": (
            "The user request requires SQL analytics."
        ),
        "execution_plan": [
            {
                "task_id": "task_1",
                "agent_name": "sql_agent",
                "task_description": (
                    "Count the number of customers "
                    "who churned."
                ),
                "depends_on": [],
            },
            {
                "task_id": "task_2",
                "agent_name": (
                    "final_response_agent"
                ),
                "task_description": (
                    "Generate the final grounded "
                    "response using the SQL result."
                ),
                "depends_on": [
                    "sql_agent",
                ],
            },
        ],
    }

    return json.dumps(response)

In [0]:
sql_agent_runners: Dict[
    str,
    AgentRunnerConfig,
] = {
    SQL_AGENT_NAME: (
        run_sql_agent,
        {
            "sql_analytics_tool": (
                mock_successful_sql_tool
            ),
        },
    ),

    FINAL_RESPONSE_AGENT_NAME: (
        run_final_response_agent,
        {},
    ),
}

In [0]:
sql_request = (
    "How many customers churned?"
)

debug_state = create_initial_state(
    user_request=sql_request,
)

debug_state = run_coordinator_agent(
    state=debug_state,
    llm_invoke=mock_sql_coordinator_llm,
)

print("COORDINATOR RESULT:")
print(
    debug_state["coordinator_result"]
)

print("\nERRORS:")
print(
    debug_state["errors"]
)

print("\nEXECUTION HISTORY:")
print(
    debug_state["execution_history"]
)

In [0]:
sql_request = (
    "How many customers churned?"
)


sql_final_state = run_multi_agent_workflow(
    user_request=sql_request,
    llm_invoke=mock_sql_coordinator_llm,
    agent_runners=sql_agent_runners,
)


In [0]:
state = run_coordinator_agent(
    state=state,
)

In [0]:
display_workflow_results(
    final_state=sql_final_state,
)

##### 24. Test 2 — Churn Prediction Request

In [0]:
prediction_request = (
    "Predict the churn risk for customer 1001."
)

prediction_final_state = run_multi_agent_workflow(
    user_request=prediction_request,
)

In [0]:
display_workflow_results(
    final_state=prediction_final_state,
)

##### 25. Test 3 — Customer Notes Request

In [0]:
vector_request = (
    "Why are customers likely to cancel the service?"
)

vector_final_state = run_multi_agent_workflow(
    user_request=vector_request,
)

In [0]:
display_workflow_results(
    final_state=vector_final_state,
)

##### 26. Test 4 — Retention Workflow

In [0]:
retention_request = (
    "For customer 1001, predict churn risk, review "
    "similar customer notes, recommend a retention "
    "action, and provide a final response."
)

retention_final_state = run_multi_agent_workflow(
    user_request=retention_request,
)

In [0]:
display_workflow_results(
    final_state=retention_final_state,
)

##### 27. Validate Execution Order

In [0]:
print("EXECUTION HISTORY")
print("=" * 100)

for record in retention_final_state[
    "execution_history"
]:
    print(record)

##### 28. Validate Agent Results

In [0]:
print("AGENT RESULT NAMES")
print("=" * 100)

for agent_name in retention_final_state[
    "agent_results"
]:
    print(agent_name)

##### 29. Validate Workflow Completion

In [0]:
workflow_completed = is_workflow_complete(
    state=retention_final_state,
)

print(
    f"Workflow completed: {workflow_completed}"
)

##### 30. Validate Errors

In [0]:
workflow_errors = retention_final_state.get(
    "errors",
    [],
)

if workflow_errors:
    print("Workflow completed with errors.")

    for error in workflow_errors:
        print(error)

else:
    print(
        "Workflow completed without errors."
    )

##### 31. Optional Test Harness

In [0]:
test_questions = [
    "How many customers churned?",

    "What is the churn rate?",

    "Predict the churn risk for customer 1001.",

    (
        "Why are customers likely to cancel the service?"
    ),

    (
        "For customer 1001, predict churn risk,  review similar customer notes, recommend a retention action, and provide a final response."
    ),
]

In [0]:
for test_number, question in enumerate(
    test_questions,
    start=1,
):
    print("\n")
    print("=" * 100)
    print(
        f"TEST {test_number}"
    )
    print("=" * 100)

    print(
        f"QUESTION:\n{question}"
    )

    try:
        test_state = run_multi_agent_workflow(
            user_request=question,
        )

        print("\nFINAL RESPONSE:")

        print(
            test_state.get(
                "final_response",
                "No final response generated.",
            )
        )

        print("\nEXECUTED AGENTS:")

        for agent_name in test_state.get(
            "agent_results",
            {},
        ):
            print(
                f"- {agent_name}"
            )

        print("\nWORKFLOW STATUS: PASS")

    except Exception as exc:
        print(
            f"\nWORKFLOW STATUS: FAIL"
        )

        print(
            f"ERROR: {str(exc)}"
        )

##### 32. Expected Results

- The Coordinator creates a valid execution plan.

- Only unfinished tasks are considered.

- Tasks execute only after their dependencies succeed.

- Each agent stores its validated result in shared state.
 
- Each completed task is skipped in future iterations.

- The workflow stops when all planned tasks complete.

- The Final Response Agent creates a grounded response.

- Blocked or invalid workflows produce clear errors.

##### 33. Key Learnings

###### Shared state coordinates the workflow

- All agents read from and write to the same MultiAgentState.

- This allows downstream agents to use earlier results without calling earlier agents directly.

###### The execution plan controls agent order

The orchestrator does not hard-code one fixed workflow.

It follows: ``` text state["execution_plan"] ``` created by the Coordinator Agent.


###### Dependencies determine readiness

A task can execute only when all agents in: ``` text task.depends_on ``` have successful results.


###### The orchestrator is not another AI agent

- The orchestrator contains deterministic Python logic.

- It:

    - checks completion
    - checks dependencies
    - finds ready tasks
    - runs agents
    - detects blocked workflows

- It does not make business decisions.

- The registry separates configuration from execution

- AGENT_RUNNERS stores:

    - agent name
    - agent function
    - required arguments

- The generic run_agent() function can therefore execute agents with different function signatures.

###### Individual agents remain responsible for their own work

- Each agent continues to handle:

    - tool execution
    - output validation
    - result storage
    - execution-history updates
    - agent-level error recording

- The orchestrator does not duplicate that logic.

##### 34. Notebook Conclusion

This notebook completes the end-to-end multi-agent customer-support workflow.

The final architecture now contains:

- Coordinator Agent
- SQL Agent
- Prediction Agent
- Vector Search Agent
- Retention Agent
- Final Response Agent
- Dependency-based Orchestrator
- Shared Multi-Agent State
- Validated Agent Schemas
- Execution History
- Error Tracking

The Coordinator decides which agents are needed and creates the execution plan.

The orchestrator reads that plan, checks task dependencies, executes ready agents, prevents duplicate execution, detects blocked workflows, and continues until all planned work is complete.

Each specialized agent remains independent and responsible for its own business logic.

The final result is a modular and extensible multi-agent system that can support SQL analytics, churn prediction, semantic customer-note retrieval, retention recommendations, and grounded customer responses.